<a href="https://colab.research.google.com/github/mangacindy9-code/plp-da-a2-rainfall-advisory/blob/main/Rainfall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rainfall Analysis and Planting Advisory
Cleaning Machakos station data, analysing rainfall for five Kenyan counties with NumPy, and producing a planting advisory for a farmer-support NGO.

In [1]:
import numpy as np
import csv

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = np.array(["Jan","Feb","Mar","Apr","May","Jun",
                   "Jul","Aug","Sep","Oct","Nov","Dec"])

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],  # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],   # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],  # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],  # Mombasa
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

## 1. Cleaning the Machakos data
The Machakos row was typed by hand. Invalid entries are caught with `except ValueError` and replaced by the rounded mean of the valid values.

In [2]:
def clean_machakos(raw):
    values, bad_idx = [], []
    for i, item in enumerate(raw):
        try:
            values.append(int(item))
        except ValueError:
            values.append(None)
            bad_idx.append(i)
    valid = [v for v in values if v is not None]
    fill = round(sum(valid) / len(valid))
    cleaned = [fill if v is None else v for v in values]
    return cleaned, bad_idx, fill

machakos_clean, imputed_idx, fill_value = clean_machakos(machakos_raw)
print("Imputed months:", [str(months[i]) for i in imputed_idx])
print("Imputed value (mm):", fill_value)
print("Cleaned Machakos row:", machakos_clean)

Imputed months: ['Apr', 'Sep']
Imputed value (mm): 68
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## 2. Combining the data into a 5 × 12 array

In [3]:
R = np.array(clean_rows + [machakos_clean])
print("Shape:", R.shape)
print("dtype:", R.dtype)

Shape: (5, 12)
dtype: int64


## 3. Annual totals and monthly averages

In [4]:
annual_totals = R.sum(axis=1)
monthly_avg = R.mean(axis=0)

for county, total in zip(counties, annual_totals.tolist()):
    print(f"{county}: {total} mm")

print()
for m, avg in zip(months.tolist(), monthly_avg.round(1).tolist()):
    print(f"{m}: {avg} mm")

Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

Jan: 47.0 mm
Feb: 47.6 mm
Mar: 91.0 mm
Apr: 164.4 mm
May: 171.2 mm
Jun: 67.6 mm
Jul: 56.6 mm
Aug: 58.2 mm
Sep: 64.2 mm
Oct: 67.8 mm
Nov: 127.4 mm
Dec: 80.6 mm


## 4. Wettest month per county

In [5]:
wettest_idx = np.argmax(R, axis=1)
wettest_names = months[wettest_idx]

for county, month in zip(counties, wettest_names.tolist()):
    print(f"{county}: {month}")

Nairobi: Apr
Nakuru: Apr
Kisumu: Apr
Mombasa: May
Machakos: Nov


## 5. Dry months (below 50 mm)

In [6]:
dry_mask = R < 50
dry_counts = dry_mask.sum(axis=1)

for county, d in zip(counties, dry_counts.tolist()):
    print(f"{county}: {d} dry months")

Nairobi: 4 dry months
Nakuru: 3 dry months
Kisumu: 0 dry months
Mombasa: 2 dry months
Machakos: 6 dry months


## 6. Planting windows (best two consecutive months)
Adding `R[:, :-1] + R[:, 1:]` gives the combined rainfall of every pair of neighbouring months. The pair with the highest sum is the recommended window.

In [7]:
pair_sums = R[:, :-1] + R[:, 1:]          # shape (5, 11)
start = np.argmax(pair_sums, axis=1)
windows = np.char.add(np.char.add(months[start], "-"), months[start + 1])

for county, w in zip(counties, windows.tolist()):
    print(f"{county}: {w}")

Nairobi: Apr-May
Nakuru: Apr-May
Kisumu: Apr-May
Mombasa: Apr-May
Machakos: Nov-Dec


## 7. Saving the summary CSV

In [8]:
with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["county", "annual_total_mm", "wettest_month",
                     "dry_months", "planting_window"])
    for row in zip(counties, annual_totals.tolist(), wettest_names.tolist(),
                   dry_counts.tolist(), windows.tolist()):
        writer.writerow(row)

with open("rainfall_summary.csv") as f:
    print(f.read())

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,Apr,4,Apr-May
Nakuru,916,Apr,3,Apr-May
Kisumu,1371,Apr,0,Apr-May
Mombasa,1075,May,2,Apr-May
Machakos,811,Nov,6,Nov-Dec



## Planting Advisory
Farmers in Nairobi, Nakuru, Kisumu and Mombasa should prepare their land in March and plant in April–May, when the two-month rainfall peaks. Kisumu is the wettest county (1,371 mm) with no dry months, so it has the lowest drought risk. Machakos is the driest (811 mm, six dry months) and its best window is November–December, so farmers there should choose short-season crops and conserve moisture. Mombasa's main peak is May, so planting should not be delayed beyond early May. Machakos data had two missing months (April and September) filled with an estimate, so that advice is less certain and should be confirmed with a local station.

## Data Quality Notes
- Machakos April ("N/A") and September (blank) were invalid and were imputed with the rounded mean of the valid months (68 mm).
- Imputation can hide a truly wet or dry month, so Machakos totals, dry-month count and planting window may be inaccurate.
- The data covers a single year, so it may not represent long-term rainfall patterns.